# Benchmark: FLUX.1-schnell on T4 16GB (local model, NF4 quantized)
Schnell needs ~24GB at full precision, does not fit T4 16GB as-is. Uses 4-bit NF4 quantization to fit. Set Runtime to T4 GPU first.

In [ ]:
!pip install -q -U diffusers transformers accelerate sentencepiece protobuf bitsandbytes

## Log in to Hugging Face
Gated model — accept access at huggingface.co/black-forest-labs/FLUX.1-schnell first, then run this and paste your token.

In [ ]:
from huggingface_hub import login
login()

In [ ]:
import torch, time, gc
from diffusers import FluxPipeline, FluxTransformer2DModel, BitsAndBytesConfig as DiffusersBnBConfig
from transformers import T5EncoderModel, BitsAndBytesConfig as TransformersBnBConfig

MODEL_ID = "black-forest-labs/FLUX.1-schnell"
load_start = time.time()

transformer = FluxTransformer2DModel.from_pretrained(
    MODEL_ID, subfolder="transformer",
    quantization_config=DiffusersBnBConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4"),
    torch_dtype=torch.bfloat16, low_cpu_mem_usage=True,
)
text_encoder_2 = T5EncoderModel.from_pretrained(
    MODEL_ID, subfolder="text_encoder_2",
    quantization_config=TransformersBnBConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4"),
    torch_dtype=torch.bfloat16, low_cpu_mem_usage=True,
)
gc.collect(); torch.cuda.empty_cache()

pipe = FluxPipeline.from_pretrained(MODEL_ID, transformer=transformer, text_encoder_2=text_encoder_2, torch_dtype=torch.bfloat16, low_cpu_mem_usage=True)
pipe.enable_model_cpu_offload()
gc.collect(); torch.cuda.empty_cache()

load_time = time.time() - load_start
print(f"Model loaded in {load_time:.1f}s")

In [ ]:
PROMPT = ("Chiku and Pinku starting their adventure in the garden. Style: Classic 1990s hand-drawn animated jungle film with clean ink outlines and cel shading. Chiku, the small sleek cat with soft gray fur. Pointed ears. Sharp green eyes. Long graceful tail. Four agile legs. Standing on grass. Pinku, the friendly dog with golden brown fur. Floppy ears. Bright loyal eyes. Wagging tail. Four legs. Running beside Chiku. Both characters visible. Jungle garden setting with lush green grass and bushes. Afternoon sunlight filtering through leaves. Light beige ground. The scene shows excitement and adventure.")

torch.cuda.reset_peak_memory_stats()
gen_start = time.time()
image = pipe(prompt=PROMPT, height=1024, width=1024, num_inference_steps=4, guidance_scale=0.0, generator=torch.Generator(device="cuda").manual_seed(42)).images[0]
gen_time = time.time() - gen_start
peak_vram_gb = torch.cuda.max_memory_allocated() / 1e9

image.save("benchmark_schnell.png")
print(f"Load: {load_time:.1f}s | Generation: {gen_time:.2f}s | Peak VRAM: {peak_vram_gb:.2f} GB")
image